# Weekly Challenge

In [1]:
!pip install torch torchvision

## 1. ResNet 모델을 불러와 새로운 이미지 데이터셋을 분류하세요.

In [2]:
import torch
import torch.nn as nn
from torch import Tensor
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights

In [3]:
# 하이퍼파라미터
h_params = {
    "img_size": 224,
    "batch_size": 128,
    "epochs": 3,
    "lr": 1e-4,
    "weight_decay": 0.05,
    "n_classes": 37,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

In [4]:
# 데이터셋
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(h_params["img_size"], interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform = ResNet50_Weights.IMAGENET1K_V1.transforms() # resize[256](BILINEAR) > CenterCrop[224] > Norm
train_dataset = datasets.OxfordIIITPet(
    root="./data",
    split="trainval", # 학습용으로 사용
    target_types="category",
    download=True,
    transform=train_transform
)
eval_dataset = datasets.OxfordIIITPet(
    root="./data",
    split="test", # validation으로 사용
    target_types="category",
    download=True,
    transform=eval_transform
)

# 데이터 로더
train_dataloader = DataLoader(
    train_dataset,
    batch_size=h_params["batch_size"],
    shuffle=True,
    drop_last=True,
    num_workers=0,
    pin_memory=True,
)
eval_dataloader = DataLoader(
    eval_dataset,
    batch_size=h_params["batch_size"],
    shuffle=False,
    drop_last=False,
    num_workers=0,
    pin_memory=True,
)

# 모델 정의
class NewResNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.head = nn.Linear(self.backbone.fc.in_features, n_classes)
        self.backbone.fc = nn.Identity()
    def forward(self, x: Tensor):
        x = self.backbone(x) # w/ avgpool
        return self.head(x)

resnet = NewResNet(n_classes=h_params["n_classes"]).to(h_params["device"]) # 모델 초기화
print(resnet(torch.rand(2, 3, 224, 224).to(h_params["device"])).shape) # 테스트

100%|██████████| 792M/792M [01:19<00:00, 9.91MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 9.94MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 187MB/s]


torch.Size([2, 37])


In [5]:
def train_one_epoch(model, criterion, dataloader, optimizer, device):
    """한 epoch 학습"""
    model.train()

    for samples, targets in dataloader:
        samples = samples.to(non_blocking=True, device=device)
        targets = targets.to(non_blocking=True, device=device)

        outputs = model(samples)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    """평가 함수"""
    model.eval()
    total_loss, correct, total = 0., 0, 0 # 손실값, 맞은 개수, 전체 샘플 수

    for samples, targets in dataloader:
        samples = samples.to(non_blocking=True, device=device)
        targets = targets.to(non_blocking=True, device=device)

        outputs = model(samples)
        loss = criterion(outputs, targets) # 배치 평균 손실값

        total_loss += loss.item() * targets.shape[0] # 손실값 전체
        correct += (outputs.argmax(dim=-1) == targets).sum().item() # scalar로 기록
        total += targets.shape[0]

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc

In [6]:
lr = h_params["lr"]
weight_decay = h_params["weight_decay"]

optimizer = torch.optim.AdamW(resnet.parameters(), lr=lr, weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()

for epoch in range(h_params["epochs"]):
    train_one_epoch(resnet, criterion, train_dataloader, optimizer, h_params["device"])
    resnet_val_loss, resnet_val_acc = evaluate(resnet, eval_dataloader, criterion, h_params["device"])

    print(f"[Epoch {epoch}] Loss: {resnet_val_loss:.4f}\tAcc: {resnet_val_acc:.4f}")

[Epoch 0] Loss: 0.7941	Acc: 0.8525
[Epoch 1] Loss: 0.4090	Acc: 0.9016
[Epoch 2] Loss: 0.3812	Acc: 0.9043


## 2. 이미지 데이터셋과 사전 훈련된 VGG16 모델을 가져와 전이 학습을 수행하세요.

In [7]:
from torchvision.models import vgg16, VGG16_Weights

In [8]:
# VGG16도 ResNet50과 같은 transform을 사용
VGG16_Weights.IMAGENET1K_V1.transforms()

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [9]:
# 모델 정의
class NewVGG16(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.backbone = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
        self.backbone.classifier[-1] = nn.Linear(self.backbone.classifier[-1].in_features, n_classes)
        # self.head = nn.Linear(self.backbone.classifier[0].in_features, n_classes)
        # self.backbone.classifier = nn.Identity()
    def forward(self, x: Tensor):
        return self.backbone(x) # w/ avgpool


vggnet = NewVGG16(n_classes=h_params["n_classes"]).to(h_params["device"]) # 모델 초기화
print(vggnet(torch.rand(2, 3, 224, 224).to(h_params["device"])).shape) # 테스트

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 215MB/s]


torch.Size([2, 37])


In [10]:
optimizer = torch.optim.AdamW(vggnet.parameters(), lr=lr, weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()

for epoch in range(h_params["epochs"]):
    train_one_epoch(vggnet, criterion, train_dataloader, optimizer, h_params["device"])
    vggnet_val_loss, vggnet_val_acc = evaluate(vggnet, eval_dataloader, criterion, h_params["device"])

    print(f"[Epoch {epoch}] Loss: {vggnet_val_loss:.4f}\tAcc: {vggnet_val_acc:.4f}")

[Epoch 0] Loss: 0.5092	Acc: 0.8307
[Epoch 1] Loss: 0.4287	Acc: 0.8605
[Epoch 2] Loss: 0.3940	Acc: 0.8858


## 3. 동일한 데이터셋에서 ResNet과 VGG16을 각각 학습시켜 성능을 비교하세요.

In [11]:
print(f"[ResNet50]")
print(f"# Parmas: {sum(param.numel() for param in resnet.parameters()) / 1e6:.1f}M Loss: {resnet_val_loss:.4f} Acc: {resnet_val_acc:.4f}")
print(f"[VGG16]")
print(f"# Parmas: {sum(param.numel() for param in vggnet.parameters()) / 1e6:.1f}M Loss: {vggnet_val_loss:.4f} Acc: {vggnet_val_acc:.4f}")

[ResNet50]
# Parmas: 23.6M Loss: 0.3812 Acc: 0.9043
[VGG16]
# Parmas: 134.4M Loss: 0.3940 Acc: 0.8858


## 4. 가상 데이터셋을 생성한 뒤, GridSearch와 RandomSearch 기법으로 하이퍼파라미터 튜닝을 진행하세요.

### Grid Search

In [12]:
from collections import defaultdict

# ResNet에 대해 (lr, weight decay) Grid Search

lrs = [5e-5, 1e-4, 5e-4]
weight_decays = [1e-8, 0.05, 0.1]

best_metrics_gs = defaultdict(float)
for lr in lrs:
    for weight_decay in weight_decays:
        resnet = NewResNet(n_classes=h_params["n_classes"]).to(h_params["device"]) # 모델 초기화
        optimizer = torch.optim.AdamW(resnet.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.CrossEntropyLoss()

        for epoch in range(h_params["epochs"]):
            train_one_epoch(resnet, criterion, train_dataloader, optimizer, h_params["device"])
            resnet_val_loss, resnet_val_acc = evaluate(resnet, eval_dataloader, criterion, h_params["device"])
            if resnet_val_acc > best_metrics_gs["acc"]: # 정확도를 기준으로 최적의 하이퍼파라미터 선택
                best_metrics_gs.update({"loss": resnet_val_loss})
                best_metrics_gs.update({"acc": resnet_val_acc})
                best_metrics_gs.update({"lr": lr})
                best_metrics_gs.update({"weight_decay": weight_decay})

            print(f"[LR: {lr:.2e} WD: {weight_decay:.2e} Epoch {epoch}]\tLoss: {resnet_val_loss:.4f} Acc: {resnet_val_acc:.4f}")

[LR: 5.00e-05 WD: 1.00e-08 Epoch 0]	Loss: 1.3127 Acc: 0.8460
[LR: 5.00e-05 WD: 1.00e-08 Epoch 1]	Loss: 0.7142 Acc: 0.8926
[LR: 5.00e-05 WD: 1.00e-08 Epoch 2]	Loss: 0.4493 Acc: 0.9092
[LR: 5.00e-05 WD: 5.00e-02 Epoch 0]	Loss: 1.3811 Acc: 0.8416
[LR: 5.00e-05 WD: 5.00e-02 Epoch 1]	Loss: 0.7004 Acc: 0.8978
[LR: 5.00e-05 WD: 5.00e-02 Epoch 2]	Loss: 0.4563 Acc: 0.9103
[LR: 5.00e-05 WD: 1.00e-01 Epoch 0]	Loss: 1.3379 Acc: 0.8452
[LR: 5.00e-05 WD: 1.00e-01 Epoch 1]	Loss: 0.7344 Acc: 0.8956
[LR: 5.00e-05 WD: 1.00e-01 Epoch 2]	Loss: 0.5014 Acc: 0.9057
[LR: 1.00e-04 WD: 1.00e-08 Epoch 0]	Loss: 0.7708 Acc: 0.8836
[LR: 1.00e-04 WD: 1.00e-08 Epoch 1]	Loss: 0.4485 Acc: 0.8940
[LR: 1.00e-04 WD: 1.00e-08 Epoch 2]	Loss: 0.4115 Acc: 0.8907
[LR: 1.00e-04 WD: 5.00e-02 Epoch 0]	Loss: 0.8278 Acc: 0.8517
[LR: 1.00e-04 WD: 5.00e-02 Epoch 1]	Loss: 0.4479 Acc: 0.8997
[LR: 1.00e-04 WD: 5.00e-02 Epoch 2]	Loss: 0.3486 Acc: 0.9071
[LR: 1.00e-04 WD: 1.00e-01 Epoch 0]	Loss: 0.7277 Acc: 0.8643
[LR: 1.00e-04 WD: 1.00e-

In [13]:
for k, v in best_metrics_gs.items():
    print(f"{k}: {v:.4f}")

acc: 0.9103
loss: 0.4563
lr: 0.0001
weight_decay: 0.0500


### Random Search

In [14]:
import random
import math

# ResNet에 대해 (lr, weight decay) Random Search
lr_range = (5e-5, 5e-4)
weight_decay_range = (1e-8, 0.1)
n_trials = 9 # Grid Search와 동일하게 설정

best_metrics_rs = defaultdict(float)
for _ in range(n_trials):
    lr = random.uniform(lr_range[0], lr_range[1])
    wd_exp = random.uniform(math.log10(weight_decay_range[0]), math.log10(weight_decay_range[1])) # 로그 스케일로 균등하게 선택
    weight_decay = 10 ** wd_exp
    resnet = NewResNet(n_classes=h_params["n_classes"]).to(h_params["device"]) # 모델 초기화
    optimizer = torch.optim.AdamW(resnet.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(h_params["epochs"]):
        train_one_epoch(resnet, criterion, train_dataloader, optimizer, h_params["device"])
        resnet_val_loss, resnet_val_acc = evaluate(resnet, eval_dataloader, criterion, h_params["device"])
        if resnet_val_acc > best_metrics_rs["acc"]: # 정확도를 기준으로 최적의 하이퍼파라미터 선택
            best_metrics_rs.update({"loss": resnet_val_loss})
            best_metrics_rs.update({"acc": resnet_val_acc})
            best_metrics_rs.update({"lr": lr})
            best_metrics_rs.update({"weight_decay": weight_decay})

        print(f"[LR: {lr:.2e} WD: {weight_decay:.2e} Epoch {epoch}]\tLoss: {resnet_val_loss:.4f} Acc: {resnet_val_acc:.4f}")

[LR: 4.33e-04 WD: 3.86e-08 Epoch 0]	Loss: 0.7860 Acc: 0.7563
[LR: 4.33e-04 WD: 3.86e-08 Epoch 1]	Loss: 0.8780 Acc: 0.7522
[LR: 4.33e-04 WD: 3.86e-08 Epoch 2]	Loss: 0.6983 Acc: 0.7956
[LR: 1.18e-04 WD: 2.32e-05 Epoch 0]	Loss: 0.6851 Acc: 0.8670
[LR: 1.18e-04 WD: 2.32e-05 Epoch 1]	Loss: 0.4361 Acc: 0.8806
[LR: 1.18e-04 WD: 2.32e-05 Epoch 2]	Loss: 0.3693 Acc: 0.9052
[LR: 3.66e-04 WD: 6.31e-02 Epoch 0]	Loss: 0.7352 Acc: 0.7762
[LR: 3.66e-04 WD: 6.31e-02 Epoch 1]	Loss: 0.7725 Acc: 0.7697
[LR: 3.66e-04 WD: 6.31e-02 Epoch 2]	Loss: 0.6631 Acc: 0.7945
[LR: 1.23e-04 WD: 2.69e-02 Epoch 0]	Loss: 0.6136 Acc: 0.8738
[LR: 1.23e-04 WD: 2.69e-02 Epoch 1]	Loss: 0.4646 Acc: 0.8817
[LR: 1.23e-04 WD: 2.69e-02 Epoch 2]	Loss: 0.3695 Acc: 0.8953
[LR: 1.49e-04 WD: 7.48e-06 Epoch 0]	Loss: 0.5718 Acc: 0.8790
[LR: 1.49e-04 WD: 7.48e-06 Epoch 1]	Loss: 0.3850 Acc: 0.8953
[LR: 1.49e-04 WD: 7.48e-06 Epoch 2]	Loss: 0.3832 Acc: 0.8866
[LR: 8.03e-05 WD: 6.08e-05 Epoch 0]	Loss: 0.8722 Acc: 0.8561
[LR: 8.03e-05 WD: 6.08e-

In [15]:
for k, v in best_metrics_rs.items():
    print(f"{k}: {v:.4f}")

acc: 0.9161
loss: 0.3562
lr: 0.0001
weight_decay: 0.0001
